# Automated Warehouse Robot Controller

This notebook outlines the design and implementation of a multi-agent control system for coordinating a fleet of warehouse robots. The system coordinates multiple robots to retrieve shelves and deliver them to packing stations without collisions or deadlocks. Various **Multi-Agent Pathfinding (MAPF)** techniques, including **Independent A***, **Cooperative A***, **Hill Climbing Optimization**, and **Conflict-Based Search (CBS)**, are employed to find collision-free paths that minimize both makespan and flowtime. The project includes comprehensive performance evaluation, deadlock detection, and visual simulation with heatmap analysis of warehouse congestion patterns.

## Data Collection & Research
We will be working on the Standard dataset by the time we collect a local one 

There are multiple choices for the standard dataset:
- **Small Grid**: 63*161
- **Medium Grid**: 84*170
- **Large Grid**: 123*321
- **Very Larg Grid**: 165*340

### Grid Envirenment Class
This class represents the warehouse as a grid-based environment that manages boundaries, obstacles, and valid robot movements.



In [1]:
class GridEnvironment:
    """
    Represents a 2D grid-based warehouse environment.
    
    Attributes:
        grid: 2D array where True = free space, False = obstacle (shelf/wall)
        height: Number of rows
        width: Number of columns
        
    Methods provide obstacle checking, neighbor finding, and visualization.
    """

    def __init__(self, filename):
        """ Load grid from file """
        self.grid = self.load_from_file(filename)
        """ Initialize grid with dimensions """
        self.height = len(self.grid)
        self.width = len(self.grid[0]) if self.height > 0 else 0

    def is_valid_position(self, x, y):
        """ Check if position is within grid bounds """
        return 0 <= x < self.width and 0 <= y < self.height

    def is_walkable(self, x, y):
        """ Check if position is walkable (not obstacle)"""
        return self.is_valid_position(x, y) and self.grid[y][x]

    def get_neighbors(self, x, y):
        """Get adjacent cells (4-directional)"""
        directions = [(0,1), (1,0), (0,-1), (-1,0)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if self.is_walkable(nx, ny):
                neighbors.append((nx, ny))

        return neighbors

    def load_from_file(self, filename):
        """ Load grid from file """
        grid = []
        with open(filename, 'r') as f:
            for line in f:
                row = []
                for char in line.strip():
                    if char == '.':
                        row.append(True)
                    elif char == 'T':
                        row.append(False)
                grid.append(row)
        return grid

    def save_to_file(self, filename):
        """ Save grid to file """
        with open(filename, 'w') as f:
            for row in self.grid:
                line = ''.join(['.' if cell else 'T' for cell in row])
                f.write(line + '\n')

    def visualize(self):
        import matplotlib.pyplot as plt
        from matplotlib.colors import ListedColormap
        import numpy as np
        """Visualize the warehouse grid"""
        grid_array = np.array(self.grid, dtype=int)
        
        fig, ax = plt.subplots(figsize=(12, 12))
        
        cmap = ListedColormap(['black', 'white'])
        ax.imshow(grid_array, cmap=cmap, origin='upper', interpolation='nearest')
        
        ax.set_xticks(np.arange(-0.5, self.width, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.height, 1), minor=True)
        
        ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.3, alpha=0.5)
        
        ax.set_xticks(np.arange(0, self.width, 20))
        ax.set_yticks(np.arange(0, self.height, 20))
        

        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(f"Warehouse Grid ({self.width}x{self.height})")
        
        plt.tight_layout()
        plt.show()

print("Grid class defnined")



Grid class defnined


## Problem Definition

Given N robots at start positions and N goal positions (shelves to retrieve), find a set of
conflict-free paths such that all robots reach their goals in minimal time.

## Problem Formulation
### 1. State Representation

Each state is represented by a dictionary with the following structure:

```python
state = {
    
    # The Robots
    'robots': [
        {
            'id': robot_id,                             # Unique identifier (0, 1, 2, ...)
            'start_position': (x0, y0),                 # Start postion (fixed)
            'goal_position': (gx, gy),                  # Goal position (fixed)
            'at_goal': bool,                            # True if positions[robot_id] == goal_position
            'color': str                                # The color of the robot, used for visualization stuff
        },
        # ... more robots
    ],
    
    # Global state
    'positions': {robot_id: (x, y)},          # Current position of the robot in the grid given its id
    'collisions': count,                      # Cumulative collision count
    'deadlock': bool,                         # True if circular wait detected
    'grid': GridEnvironment                   # Reference to warehouse layout
}
```

### 2. Goal Test

**Primary Goal**:
- No collisions occurred
- No deadlock detected
- All robots at their goal positions


**Success Criteria: (Detail)**
1. All robots reach goals 
2. No collisions/conflicts 
3. No deadlocks 
4. Minimum makespan  (secondary objective)
5. Minimum flowtime  (secondary objective)

### 3. Actions
- All robots move simultaneously, some could wait

- SubActions per robot: 
    - MoveUp: move up if walkable (path = [..., (x, y), (x - 1, y), ...]) 


    - MoveDown: move down if walkable (path = [..., (x, y), (x + 1, y), ...]) 


    - MoveLeft: move left if walkable (path = [..., (x, y), (x, y - 1), ...]) 


    - MoveRight: move right if walkable (path = [..., (x, y), (x, y + 1), ...]) 


    - Wait: stay in current position (path = [..., (x, y), (x, y), ...]) 



### 4. Transition Model
- The new state after applying the action:
    - The new position for each robot is added to its path, if it is a wait just add the last position to the path i.e. [..., (x, y), (x, y), ...]

    - Each robot may reach its goal_position, hence the attribute 'at_goal' may be updated 

    - The positions of the robots is changed

    - The number of collisions in the state is updated

    - The deadlock state is update (a deadlock may occur)

### 5. Path Cost

- Each move costs: 1 time unit
- Each wait costs: 1 time unit


### Initial State


**Properties of Initial State:**
- All robots at distinct start positions
- All start positions are walkable
- No paths planned yet
- No conflicts exist initially
- Time counter starts at 0
- Cost g(n) = 0 (no moves yet)


**Initial State Example:**
```python
initial_state = {
    'robots': [
        {'id': 1, 'start_position': (1, 1), 'goal_position': (8, 8), 'at_goal': False},
        {'id': 2, 'start_position': (8, 1), 'goal_position': (1, 8), 'at_goal': False},
        {'id': 3, 'start_position': (1, 8), 'goal_position': (8, 1), 'at_goal': False},
    ],
    'positions': {1: (1, 1), 2: (8, 1), 3: (1, 8)}
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment(filename)
}
```

- Robot 1: starts at (1,1), goal (8,8)
- Robot 2: starts at (8,1), goal (1,8)
- Robot 3: starts at (1,8), goal (8,1)
- Robots' positions are their starting positions
- No collisions 
- No deadlock
- Grid loaded from the file


### Robot Class

This class represents a single robot in the grid, managing its position, goal, and movement within the environment.

In [2]:
class Robot :
    """ Represents a single robot """
    def __init__(self, robot_id: int, start_position: tuple, goal_position: tuple, color: str):
        """
        Args:
            grid (GridEnvironment): The grid this robot moves in
            start_pos (tuple): Starting (x, y) position
            goal_pos (tuple): Goal (gx, gy) position
            color (str): Robot color
        """

        self.id = robot_id
        self.start_pos = start_position
        self.goal_pos = goal_position
        self.color = color

    def get_start_position(self) -> tuple:
        """Get start position"""
        return self.start_position

    def get_goal_position(self) -> tuple:
        """Get goal position """
        return self.goal_position

    def get_color(self) -> str:
        """Get color"""
        return self.color
    
print("Robot class defined")

Robot class defined


### Node Class
Now, let's define the Node class which will represent states in the search space

In [3]:
class Node:
    """
    Represents a node in the search tree.
    
    Attributes:
        state: The current state (configuration of all robots)
        parent: Parent node in search tree
        action: Action that led to this node
        g: Cumulative cost (actual path cost from start)
        h: Heuristic value (estimated cost to goal)
        f: Total evaluation cost (g + h)
        depth: Depth in search tree
    """

    def __init__(self, state, parent=None, action=None, g=0, h=0):
        """
        Initialize a search node.
        
        Args:
            state: State configuration (dict of robot positions at each time)
            parent: Parent node
            action: Action to reach this node
            g: Cost from start to this node
            h: Heuristic estimate to goal
        """
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  # Actual cost
        self.h = h  # Heuristic estimate
        self.f = g + h  # f(n) = g(n) + h(n)
        self.depth = 0 if parent is None else parent.depth + 1

    def __hash__(self):
        """
        Make node hashable by hashing the state.
        Since state is a dict, convert to a canonical string form.
        """
        return hash(str(sorted(self.state.items()))) # ⚠️⚠️⚠️⚠️⚠️⚠️ We need to consider the list of robots here, lists are imutable

    def __eq__(self, other):
        """
        Two nodes are equal if their states are identical.
        """
        return self.state == other.state    

    def __gt__(self, other):
        """
        Compare this node with another node based on the evaluation function (f).

        Input Parameters:
            - other: Another Node instance.

        Output:
            - True if this node's f is greater than the other's f, else False.
        """
        return isinstance(other, Node) and self.f > other.f
    
    def __repr__(self):
        return f"Node(depth={self.depth}, g={self.g}, h={self.h}, f={self.f})"
    
print("Class Node definied")

Class Node definied


### Candidate Class
Now, we define the Candidate class which will represent a candidate solution in a local search context

In [4]:
class Candidate:
    """
    Represents a candidate solution in a local search context.

    Attributes:
        state: The specific configuration of the solution (e.g., a tour permutation, queen positions).
        value: The evaluation score of the state (lower is generally better in minimization problems).
    """
    def __init__(self, state, value):
        self.state = state
        self.value = value

    def __repr__(self):
        pass

print("Class Candidate defined")

Class Candidate defined


### Problem Class
Now, let's define the main Problem class that will encapsulate our MAPF problem

In [5]:
import copy
class AutomatedWarehouseRobotControllerProblem:
    """
    """
    def __init__(self, initial_state):
        """
        Initialize the MAPF problem.
        
        Args:
            grid: Grid object representing the environment
            robots: List of Robot objects
            max_time: Maximum time steps allowed (prevents infinite loops)
        """
        self.state = initial_state

    def is_goal(self) -> bool:
        pass

    def get_valid_actions(self, state) -> list[dict[int: tuple]]:
        """Returns possible actions for the given state, as th next postions of the robots"""
        pass

    def apply_action(self, state, action):
        """Applies the next moves that are in the action to the given state and returns the new state"""
        pass

    def expand_node(self, node) -> list[Node]:
        """Returns the child nodes of the given node"""
        pass

    def generate_neighbors(self, state):
        """Returns the neighbors of the current state, will be used in local search'Hill climbing optimization'"""
        pass

    def get_makespan(self, state) -> int:
        pass

    def get_flowtime(self, state) -> int:
        pass
    
    # We need functions that detect if the appllying an actions will lead to a deadlock, and to detect the vertex and edge collisions

print("Class AutomatedWarehouseRobotControllerProblem defined")

Class AutomatedWarehouseRobotControllerProblem defined


### Search Algorithms

In [6]:
class A_Star:
    def __init__(self, problem, strategy="cooperative"):
        pass


class Hill_Climbing:
    def __init__(self, problem):
        pass

class Conflict_Based:
    def __init__(self, problem):
        pass

### Comparative Evaluation

### Deliverables